## Data acquisition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pylablib.devices import Thorlabs

EXPOSURE_SECONDS = 0.010

print("Initializing Thorlabs camera...")
cam = Thorlabs.ThorlabsTLCamera()
raw_frame = None

try:
    # Spectral extraction needs the untouched 2D Bayer mosaic.
    cam.set_color_format("raw", color_space="linear")
    cam.set_exposure(EXPOSURE_SECONDS)

    device_info = cam.get_device_info()
    sensor_info = cam.get_sensor_info()
    color_info = cam.get_color_info()
    width, height = cam.get_detector_size()

    print(f"Model: {device_info.model}; serial: {device_info.serial_number}")
    print(f"Detector: {width} x {height}; {sensor_info.bit_depth}-bit {sensor_info.sensor_type}")

    if sensor_info.sensor_type != "bayer" or color_info is None:
        raise RuntimeError("A Bayer color camera is required for RGB spectral extraction.")

    bayer_phase = color_info.filter_array_phase
    print(f"Reported Bayer phase: {bayer_phase}")

    print("Capturing raw frame...")
    cam.start_acquisition()
    cam.wait_for_frame(timeout=max(2.0, 5 * EXPOSURE_SECONDS))
    raw_frame = cam.read_newest_image()
    if raw_frame is None:
        raise RuntimeError("The camera reported a frame, but no unread image was available.")
finally:
    cam.close()
    print("Camera connection closed safely.")

if raw_frame.ndim != 2:
    raise RuntimeError(f"Expected a 2D raw Bayer frame, received shape {raw_frame.shape}.")
if raw_frame.shape != (height, width):
    raise RuntimeError(f"Frame shape {raw_frame.shape} does not match detector {(height, width)}.")

max_count = (1 << sensor_info.bit_depth) - 1
print(f"Array shape: {raw_frame.shape}")
print(f"Data type: {raw_frame.dtype}")
print(f"Maximum pixel intensity: {np.max(raw_frame)} / {max_count}")

plt.figure(figsize=(10, 6))
plt.imshow(raw_frame, cmap="gray", vmin=0, vmax=max_count)
plt.title("Raw Unprocessed 2D Camera Frame")
plt.colorbar(label=f"Counts (0-{max_count})")
plt.show()


## Calibration phase

### Dark frame subtraction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pylablib.devices import Thorlabs
import time

# --- CONFIGURATION MATCHING ---
# IMPORTANT: This must match the exact exposure time of your science target!
EXPOSURE_SECONDS = 0.010  # 0.010s = 10ms
NUM_FRAMES_TO_AVERAGE = 10  # Stacking frames creates a super-clean baseline master dark

print("Initializing Thorlabs Camera for Dark Frame Capture...")
cam = Thorlabs.ThorlabsTLCamera()

try:
    # Configure the exact same exposure settings
    cam.set_color_format("raw", color_space="linear")
    cam.set_exposure(EXPOSURE_SECONDS)
    width, height = cam.get_detector_size()
    bit_depth = cam.get_sensor_info().bit_depth
    
    print(f"Camera ready. Target Exposure: {EXPOSURE_SECONDS*1000} ms.")
    print("⚠️  CRITICAL: Ensure the lens cap is securely on the slit before continuing! ⚠️")
    input("Press Enter once the spectrometer is completely dark to begin capture...")
    
    # Initialize a blank 32-bit float array to accumulate frames without overflowing
    dark_accumulator = np.zeros((height, width), dtype=np.float32)
    
    print(f"Collecting {NUM_FRAMES_TO_AVERAGE} dark frames for averaging...")
    cam.start_acquisition()
    
    for i in range(NUM_FRAMES_TO_AVERAGE):
        cam.wait_for_frame(timeout=max(2.0, 5 * EXPOSURE_SECONDS))
        raw_frame = cam.read_newest_image()
        if raw_frame is None:
            raise RuntimeError("No unread dark frame was available.")
        dark_accumulator += raw_frame.astype(np.float32)
        print(f"  Captured frame {i+1}/{NUM_FRAMES_TO_AVERAGE}")
        
    cam.stop_acquisition()
    
    # Keep the averaged baseline as float32 to preserve sub-count precision.
    master_dark = dark_accumulator / NUM_FRAMES_TO_AVERAGE
    print("Master dark frame generated successfully.")

finally:
    # Always guarantee safe closure of the USB stream
    cam.close()
    print("Camera connection closed safely.")

# --- SAVE FOR PHASE 2 ENGINE ---
# Save the matrix to a binary file on your disk
np.save("master_dark_10ms.npy", master_dark)
print("Saved baseline to disk as 'master_dark_10ms.npy'")

# --- VISUAL VERIFICATION ---
print(f"Mean Dark Current Level: {np.mean(master_dark):.2f} Counts")
print(f"Max Hot Pixel Level: {np.max(master_dark)} / {(1 << bit_depth) - 1}")

plt.figure(figsize=(10, 6))
plt.imshow(master_dark, cmap='inferno') # Inferno highlights tiny thermal variances
plt.title(f"Master Dark Frame ({EXPOSURE_SECONDS*1000}ms baseline)")
plt.colorbar(label="Thermal Counts")
plt.show()

### Spatial to 1D reduction

* Slicing ROI (region of interest)
* Binning rows vertically

In [ ]:
# raw_frame and bayer_phase come from the acquisition cell.
master_dark = np.load("master_dark_10ms.npy")
if master_dark.shape != raw_frame.shape:
    raise ValueError("The master dark and science frame dimensions do not match.")

# Convert before subtraction so values below the dark level do not wrap.
clean_frame = np.clip(
    raw_frame.astype(np.float32) - master_dark.astype(np.float32), 0, None
)

ROI_ROWS = slice(500, 540)  # Replace after inspecting the actual spectral ribbon.
ribbon = clean_frame[ROI_ROWS, :]

# Translate the camera-reported phase into the red and blue offsets of each 2x2 tile.
phase_offsets = {
    "red": ((0, 0), (1, 1)),
    "blue": ((1, 1), (0, 0)),
    "green_left_of_red": ((0, 1), (1, 0)),
    "green_left_of_blue": ((1, 0), (0, 1)),
    "green_left_or_red": ((0, 1), (1, 0)),
    "green_left_or_blue": ((1, 0), (0, 1)),
}
if bayer_phase not in phase_offsets:
    raise ValueError(f"Unsupported Bayer phase: {bayer_phase}")

(r_row, r_col), (b_row, b_col) = phase_offsets[bayer_phase]
roi_start = 0 if ROI_ROWS.start is None else ROI_ROWS.start
r_row = (r_row - roi_start) % 2
b_row = (b_row - roi_start) % 2
r_samples = ribbon[r_row::2, r_col::2]
b_samples = ribbon[b_row::2, b_col::2]

green_offsets = [
    (row, col)
    for row in (0, 1)
    for col in (0, 1)
    if (row, col) not in {(r_row, r_col), (b_row, b_col)}
]
g_samples = np.mean(
    [ribbon[row::2, col::2] for row, col in green_offsets], axis=0
)

r_profile_1d = np.sum(r_samples, axis=0)
g_profile_1d = np.sum(g_samples, axis=0)
b_profile_1d = np.sum(b_samples, axis=0)

# Preserve each channel's physical sensor-column coordinates for calibration.
r_pixel_columns = np.arange(r_profile_1d.size) * 2 + r_col
g_pixel_columns = np.arange(g_profile_1d.size) * 2 + 0.5
b_pixel_columns = np.arange(b_profile_1d.size) * 2 + b_col


## Data processing